## AutoCast encoder-processor-decoder model API Exploration

This notebook aims to explore the end-to-end API.


### Example dataaset

We use the `AdvectionDiffusion` dataset as an example dataset to illustrate training and evaluation of models. This dataset simulates the advection-diffusion equation in 2D.


In [1]:
import pickle
from pathlib import Path

from autoemulate.simulations.advection_diffusion import AdvectionDiffusion
from autoemulate.simulations.reaction_diffusion import ReactionDiffusion

from autocast.data.advection_diffusion import (
    AdvectionDiffusion as AdvectionDiffusionMultichannel,
)
from autocast.metrics import MAE, MSE, RMSE

THE_WELL = False
n_steps_input = 4
n_steps_output = 1
stride = 1
rollout_stride = n_steps_output
simulation_name = "reaction_diffusion"

### Read combined data into datamodule

In [2]:
from autocast.data.utils import get_datamodule

datamodule = get_datamodule(
    the_well=THE_WELL,
    simulation_name=simulation_name,
    n_steps_input=n_steps_input,
    n_steps_output=n_steps_output,
    stride=stride,
    n_train=20,
    n_valid=5,
    n_test=5
)

Loading cached simulation data from ../datasets/tmp/reaction_diffusion


### Set-up logging

In [3]:

from autocast.logging import maybe_watch_model
from autocast.logging.wandb import create_notebook_logger

logger, watch = create_notebook_logger(
    project="autocast-notebooks",
    name=f"00_01_exploration_{simulation_name}",
    tags=["notebook", simulation_name],
)

### Example shape and batch


In [4]:
batch = next(iter(datamodule.train_dataloader()))



In [5]:
n_constant_scalars = batch.constant_scalars.shape[-1]

In [6]:
from autocast.decoders.channels_last import ChannelsLast
from autocast.encoders.permute_concat import PermuteConcat
from autocast.models.encoder_decoder import EncoderDecoder
from autocast.models.encoder_processor_decoder import EncoderProcessorDecoder
from autocast.processors.vit import AViTProcessor
from autocast.utils import get_optimizer_config

batch = next(iter(datamodule.train_dataloader()))
n_channels = batch.input_fields.shape[-1]

# Define model components
encoder = PermuteConcat(
    in_channels=n_channels, n_steps_input=n_steps_input, with_constants=True
)
decoder = ChannelsLast(output_channels=n_channels, time_steps=n_steps_output)
processor = AViTProcessor(
    in_channels=(n_channels + n_constant_scalars) * n_steps_input,
    out_channels= n_channels * n_steps_output,
    spatial_resolution=(32, 32),
    hidden_dim=128,
    num_heads=8,
    n_layers=8,
    groups=8,
    patch_size=4
)


model = EncoderProcessorDecoder(
    encoder_decoder=EncoderDecoder(encoder=encoder, decoder=decoder),
    processor=processor,
    train_in_latent_space=False,
    optimizer_config=get_optimizer_config(5e-4),
    test_metrics = [MSE(), MAE(), RMSE()],
    strie = stride,
    loss_func=processor.loss_func
)
maybe_watch_model(logger, model, watch)

In [7]:
encoder.encode(batch).shape

torch.Size([16, 16, 32, 32])

In [8]:
model(batch).shape

torch.Size([16, 1, 32, 32, 2])

### Run trainer


In [9]:
#logger.logging.wandb.enabled=False

In [10]:
import lightning as L

device = "mps"  # "cpu"
# device = "cpu"
trainer = L.Trainer(
    max_epochs=30, accelerator=device, log_every_n_steps=10, logger=logger
)
trainer.fit(model, datamodule)
trainer.save_checkpoint(f"./{simulation_name}_{processor_name}_model.ckpt")

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
wandb: Currently logged in as: pconti (turing-core) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



  | Name            | Type             | Params | Mode 
-------------------------------------------------------------
0 | encoder_decoder | EncoderDecoder   | 0      | train
1 | processor       | AViTProcessor    | 1.6 M  | train
2 | loss_func       | MSELoss          | 0      | train
3 | val_metrics     | MetricCollection | 0      | train
4 | test_metrics    | MetricCollection | 0      | train
-------------------------------------------------------------
1.6 M     Trainable params
0         Non-trainable params
1.6 M     Total params
6.513     Total estimated model params size (MB)
134       Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=30` reached.


### Run the evaluation


In [11]:
trainer.test(model, datamodule)

Testing: |          | 0/? [00:00<?, ?it/s]

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_loss          6.971922266529873e-05
        test_mae           0.0063705239444971085
        test_mse           6.971921538934112e-05
        test_rmse          0.008272642269730568
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


[{'test_loss': 6.971922266529873e-05,
  'test_mae': 0.0063705239444971085,
  'test_mse': 6.971921538934112e-05,
  'test_rmse': 0.008272642269730568}]

### Example rollout


In [12]:
# A single element is the full trajectory
batch = next(iter(datamodule.rollout_test_dataloader()))

In [13]:
# First n_steps_input are inputs
print(batch.input_fields.shape)
# Remaining n_steps_output are outputs
print(batch.output_fields.shape)

torch.Size([4, 4, 32, 32, 2])
torch.Size([4, 96, 32, 32, 2])


In [14]:
# Run rollout on one trajectory
model.max_rollout_steps = 100
preds, trues = model.rollout(batch, stride=n_steps_output, free_running_only=True, max_rollout_steps=90)

print(preds.shape)
assert trues is not None
print(trues.shape)


Rollout: 100%|██████████| 90/90 [00:02<00:00, 36.84it/s]

torch.Size([4, 90, 32, 32, 2])
torch.Size([4, 90, 32, 32, 2])


In [15]:
from autocast.metrics import MSE

assert trues is not None
assert preds.shape == trues.shape
mse = MSE()
mse_error_spatial = mse(preds, trues)
mse_error = mse(preds, trues)
print("MSE spatial has shape (B,T,C):", mse_error_spatial.shape)
print("MSE overall is a single scalar:", mse_error.shape)

MSE spatial has shape (B,T,C): torch.Size([])
MSE overall is a single scalar: torch.Size([])


In [ ]:
from IPython.display import HTML

from autocast.utils import plot_spatiotemporal_video

batch_idx = 0
if simulation_name == "advection_diffusion_multichannel":
    channel_names = ["vorticity", "veldocity_x", "velocity_y", "streamfunction"]
elif simulation_name == "advection_diffusion":
    channel_names = ["vorticity"]
elif simulation_name == "reaction_diffusion":
    channel_names = ["U", "V"]
else:
    channel_names = None

anim = plot_spatiotemporal_video(
    pred=preds,
    true=trues,
    batch_idx=batch_idx,
    save_path=f"{simulation_name}_{batch_idx:02d}.mp4",
    colorbar_mode="column",
    channel_names=channel_names,
)
HTML(anim.to_jshtml())

Video saved to reaction_diffusion_00.mp4


In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.plot(preds[0,:,0,0,0].detach().numpy())
plt.plot(trues[0,:,0,0,0].detach().numpy())